# Advanced Python Custom Exceptions — 20 Worked Problems and Verified Solutions

**Based on the supplied Custom Exceptions lesson; expanded into an independent, runnable practice notebook.**

The original lesson introduces subclassing `Exception`, hierarchies, `args`/`repr`, custom attributes, API-to-HTTP mappings, JSON responses, logging, and multiple inheritance. Here, those ideas are developed into **20 distinct problems**, realistic implementations, and executable checks, including exception chaining, transaction safety, retries, asynchronous cancellation, `ExceptionGroup`, serialization, and an end-to-end order-processing capstone.

**Environment:** Python **3.11+**, Jupyter/IPython kernel, standard library only; no network, database, credentials, or external services. Run **Kernel → Restart & Run All**. Solutions precede their check cells; problems are numbered and classes use distinct names to avoid accidental redefinitions. Every intentional exception is caught by a check. Tests print a visible `PASS` marker. The last cell is an overall workbook check.

**Best-practice corrections to the source:** use built-in names sparingly (avoid defining a new `TimeoutError` when the built-in already exists); distinguish `401 Unauthorized` (missing/invalid authentication) from `403 Forbidden` (authenticated but insufficient permissions); separate internal diagnostics from public error messages; prefer Python's `logging` module over printing inside exception objects; and generally avoid multiple inheritance from built-in exception types because some have incompatible C implementations. The single-inheritance alternative is covered explicitly.

### Table of contents

- **Foundations (01–05):** hierarchies; structured context; precise validation; cause chaining; privacy-safe failures.
- **Control flow (06–08):** `else`/`finally`; transactional context managers; read-only property errors.
- **API design (09–11):** public/internal separation; transport mapping; structured logging.
- **Reliability (12–15):** bounded retry; `ExceptionGroup`; concurrent jobs; async cancellation.
- **Advanced contracts (16–19):** serialization; exception compatibility; fail-open/fail-closed; deterministic edge-case testing.
- **Integration (20):** transactional order service with HTTP mapping, retry, and regression checks.

> Work the challenge before revealing/running its solution if using this as an assignment. These examples are educational in-memory simulations; the production architecture notes mark where a real framework, durable database, or metrics service would differ.

In [1]:
import sys
import unittest

assert sys.version_info >= (3, 11), 'Python 3.11+ is required for ExceptionGroup and except*.'
CHECK = unittest.TestCase()
PASS_COUNT = 0

def passed(problem_number: int) -> None:
    """Give a visible checkpoint after all checks for one problem passed."""
    global PASS_COUNT
    PASS_COUNT += 1
    print(f'PASS — problem {problem_number:02d}')

print('Python:', sys.version.split()[0])
print('Ready: standard-library-only exercises.')


Python: 3.13.7
Ready: standard-library-only exercises.


## Problem 01 — Design a catchable library hierarchy

**Challenge.** Design a scraper exception taxonomy with one application root, networking and parsing branches, and distinct DNS and timeout leaf types. Demonstrate that leaf exceptions can be caught narrowly or via the library root without catching unrelated programming errors. Show a deliberately broad handler placed *after* a specific handler.

**Solution approach and design choices.** All application-defined failures derive from `Exception` via a single library root. A caller can catch a leaf when recovery is specific, a branch for network problems, or the root at a service boundary. Put the most specific `except` clauses first: a broad handler placed first would make narrow clauses unreachable in practice. Do not shadow Python's built-in `TimeoutError`.

In [2]:
class ScraperError(Exception):
    """Root of predictable errors raised by our scraper library."""

class ScraperNetworkError(ScraperError):
    """Network access failed."""

class ScraperDNSError(ScraperNetworkError):
    """A host name could not be resolved."""

class ScraperTimeoutError(ScraperNetworkError):
    """A scraper network operation exceeded its deadline."""

class ScraperParseError(ScraperError):
    """A response did not contain the expected product data."""

def fetch_product(mode: str) -> dict[str, str]:
    """Simulate a fetch; no real requests are performed."""
    if mode == 'dns':
        raise ScraperDNSError('Unable to resolve example.invalid')
    if mode == 'timeout':
        raise ScraperTimeoutError('Connection deadline exceeded')
    if mode == 'parse':
        raise ScraperParseError('Missing required price element')
    if mode == 'bug':
        raise TypeError('Unexpected library programming error')
    return {'sku': 'SKU-1', 'price': '15.00'}

def categorize_fetch(mode: str) -> str:
    try:
        fetch_product(mode)
    except ScraperDNSError:
        return 'dns-specific'
    except ScraperNetworkError:
        return 'other-network'
    except ScraperError:
        return 'other-scraper'
    else:
        return 'success'


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [3]:
assert issubclass(ScraperDNSError, ScraperNetworkError)
assert issubclass(ScraperTimeoutError, ScraperError)
assert not issubclass(TypeError, ScraperError)
assert categorize_fetch('dns') == 'dns-specific'
assert categorize_fetch('timeout') == 'other-network'
assert categorize_fetch('parse') == 'other-scraper'
assert categorize_fetch('ok') == 'success'
with CHECK.assertRaises(TypeError):
    categorize_fetch('bug')  # An unrelated bug is NOT turned into a library failure.
passed(1)


PASS — problem 01


**Further challenge.** Add an `HTTPError` branch and decide which leaf failures are safe to retry.

## Problem 02 — Carry structured context without breaking args

**Challenge.** Build `InventoryError` and `OutOfStockError`. The leaf should carry `sku`, `requested`, and `available` fields, preserve a meaningful base `Exception.args`, and produce a useful message. Reject negative counts before constructing the exception.

**Solution approach and design choices.** A custom exception can have typed attributes and an explicit initializer; call `super().__init__(message)` so `str`, `repr`, and `args` behave consistently. Structure is more useful than parsing text downstream. Avoid embedding passwords or personal data in exception messages.

In [4]:
class InventoryError(Exception):
    """Root for ordinary inventory domain failures."""

class OutOfStockError(InventoryError):
    """There is insufficient stock for the requested SKU."""

    def __init__(self, sku: str, requested: int, available: int) -> None:
        if not sku:
            raise ValueError('sku must not be empty')
        if any(type(n) is not int or n < 0 for n in (requested, available)):
            raise ValueError('counts must be nonnegative integers (not bool)')
        self.sku = sku
        self.requested = requested
        self.available = available
        super().__init__(
            f'Insufficient stock for {sku}: requested={requested}, available={available}'
        )

def reserve_stock(sku: str, amount: int, stock: dict[str, int]) -> int:
    if type(amount) is not int or amount < 0:
        raise ValueError('amount must be a nonnegative integer')
    available = stock.get(sku, 0)
    if amount > available:
        raise OutOfStockError(sku, amount, available)
    stock[sku] = available - amount
    return stock[sku]


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [5]:
warehouse = {'PENCIL': 3}
with CHECK.assertRaises(OutOfStockError) as captured:
    reserve_stock('PENCIL', 5, warehouse)
err = captured.exception
assert err.sku == 'PENCIL' and err.requested == 5 and err.available == 3
assert err.args == (str(err),)
assert 'OutOfStockError' in repr(err)
assert warehouse == {'PENCIL': 3}  # Failed reservation has no side effect.
assert reserve_stock('PENCIL', 2, warehouse) == 1
with CHECK.assertRaises(ValueError):
    reserve_stock('PENCIL', True, warehouse)
with CHECK.assertRaises(ValueError):
    OutOfStockError('', 1, 0)
passed(2)


PASS — problem 02


## Problem 03 — Distinguish type errors from domain errors

**Challenge.** Implement `parse_quantity(value)` accepting nonnegative integer values or digit-only strings with surrounding whitespace, but not booleans, floats, signed strings, or values over 1,000. Use `TypeError` for unsupported input types and a dedicated `ValueError` subclass for invalid values. Add table-driven boundary tests.

**Solution approach and design choices.** An exception type is part of an API contract: type mismatches and invalid *values* should not be conflated. Because `bool` subclasses `int`, use `type(value) is int` when boolean quantities are invalid. The custom error inherits from `ValueError` (a single parent) to support normal callers without fragile multiple inheritance.

In [6]:
class InvalidQuantityError(ValueError):
    """A supported quantity representation is outside the allowed domain."""

MAX_QUANTITY = 1_000

def parse_quantity(value: object) -> int:
    if type(value) is int:
        result = value
    elif isinstance(value, str):
        token = value.strip()
        if not token or not token.isascii() or not token.isdecimal():
            raise InvalidQuantityError('quantity text must contain ASCII decimal digits')
        result = int(token)
    else:
        raise TypeError('quantity must be an int or ASCII decimal string')
    if not 0 <= result <= MAX_QUANTITY:
        raise InvalidQuantityError(f'quantity must be between 0 and {MAX_QUANTITY}')
    return result


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [7]:
for raw, expected in [(0, 0), (1000, 1000), (' 007 ', 7), ('0', 0)]:
    assert parse_quantity(raw) == expected
for raw in [-1, 1001, '-1', '+2', '1.0', '', '  ', '１２']:
    with CHECK.assertRaises(InvalidQuantityError):
        parse_quantity(raw)
for raw in [True, 1.0, None, [], False]:
    with CHECK.assertRaises(TypeError):
        parse_quantity(raw)
assert issubclass(InvalidQuantityError, ValueError)
passed(3)


PASS — problem 03


## Problem 04 — Preserve a low-level cause with `raise ... from`

**Challenge.** Convert a decimal price from an external catalog. A bad numeric string should cause a domain-level `PriceFormatError`, with the original decimal conversion failure accessible via `__cause__`. Leave a valid decimal intact; reject NaN, infinity, and negative prices.

**Solution approach and design choices.** Use explicit exception chaining only when translating an error into a useful higher-level abstraction. Catch the narrow `decimal.InvalidOperation` rather than everything. An invalid decimal is different from a numeric but disallowed price, so domain validation raises without an artificial conversion cause.

In [8]:
from decimal import Decimal, InvalidOperation

class PriceFormatError(ValueError):
    """External price text cannot be parsed into a decimal amount."""

class PriceDomainError(ValueError):
    """A parsed price is nonfinite or negative."""

def parse_price(raw: str) -> Decimal:
    if not isinstance(raw, str):
        raise TypeError('price must be a string')
    try:
        price = Decimal(raw)
    except InvalidOperation as original:
        raise PriceFormatError('catalog price has invalid numeric format') from original
    if not price.is_finite() or price < 0:
        raise PriceDomainError('catalog price must be finite and nonnegative')
    return price


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [9]:
assert parse_price('0.00') == Decimal('0.00')
assert parse_price('12.345') == Decimal('12.345')
with CHECK.assertRaises(PriceFormatError) as captured:
    parse_price('not-money')
assert isinstance(captured.exception.__cause__, InvalidOperation)
assert captured.exception.__suppress_context__ is True
for raw in ['NaN', 'Infinity', '-0.01']:
    with CHECK.assertRaises(PriceDomainError):
        parse_price(raw)
with CHECK.assertRaises(TypeError):
    parse_price(1.2)
passed(4)


PASS — problem 04


## Problem 05 — Hide sensitive dependency details at the public boundary

**Challenge.** A fake credential provider raises a low-level exception whose string mentions a secret. Translate it into a safe `AuthenticationFailure` using `raise ... from None` for the public traceback, while never putting the secret in the high-level message. Verify the underlying error remains in `__context__` and that it is suppressed when formatted.

**Solution approach and design choices.** `from None` hides an implementation traceback in normal presentation, but DOES NOT erase the underlying context. It is not data sanitization by itself: explicitly construct a safe public message, and avoid returning/logging raw exception objects to callers.

In [10]:
import traceback

class CredentialProviderError(Exception):
    """Private dependency failure; never returned directly to a client."""

class AuthenticationFailure(Exception):
    """Safe public authentication failure."""

def authenticate(token: str) -> str:
    try:
        if token != 'valid-token':
            raise CredentialProviderError('credential backend rejected secret=TOP-SECRET')
        return 'user-123'
    except CredentialProviderError:
        raise AuthenticationFailure('Authentication failed') from None


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [11]:
assert authenticate('valid-token') == 'user-123'
with CHECK.assertRaises(AuthenticationFailure) as captured:
    authenticate('bad-token')
error = captured.exception
assert 'TOP-SECRET' not in str(error)
assert isinstance(error.__context__, CredentialProviderError)
assert error.__cause__ is None
assert error.__suppress_context__ is True
visible = ''.join(traceback.format_exception(error))
assert 'TOP-SECRET' not in visible
passed(5)


PASS — problem 05


**Further challenge.** `from None` does not remove sensitive data from the process. Design a logging/redaction policy for the original failure.

## Problem 06 — Prove the behavior of else and finally

**Challenge.** Implement a safe converter that records which `try`, `except`, `else`, and `finally` blocks ran. An integer input returns its doubled value. Invalid input raises `ValueError`, caught at this boundary to return `None`. Show that `finally` executes in both cases and `else` only after success.

**Solution approach and design choices.** The `else` block is deliberately narrow: it executes only if the `try` suite completed without an exception. `finally` handles cleanup and executes even on failure. Never return from `finally`, because it can override a pending exception or return value.

In [12]:
def convert_and_trace(raw: str) -> tuple[int | None, list[str]]:
    events: list[str] = []
    result: int | None = None
    try:
        events.append('try')
        result = int(raw) * 2
    except ValueError:
        events.append('except')
        result = None
    else:
        events.append('else')
    finally:
        events.append('finally')
    return result, events


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [13]:
assert convert_and_trace('21') == (42, ['try', 'else', 'finally'])
assert convert_and_trace('oops') == (None, ['try', 'except', 'finally'])
with CHECK.assertRaises(TypeError):
    convert_and_trace(None)  # Not caught by the narrower ValueError handler.
passed(6)


PASS — problem 06


## Problem 07 — Write a transaction context manager with rollback

**Challenge.** Implement a dictionary-backed transaction with a `contextlib.contextmanager`. Mutations inside the block commit on success and are rolled back if an exception occurs. Ensure a failure does not get swallowed. Include a test where a preexisting value is changed before failure.

**Solution approach and design choices.** Take a snapshot before the operation. In a real database, rely on the database's atomic transaction protocol, not `dict.copy()`; this shallow snapshot assumes flat immutable values. Catch `Exception` at this resource boundary *only* to restore the snapshot, then use bare `raise` to preserve traceback and identity. Do not intercept `KeyboardInterrupt` or `SystemExit`.

In [14]:
from contextlib import contextmanager
from collections.abc import Iterator

@contextmanager
def dictionary_transaction(data: dict[str, int]) -> Iterator[dict[str, int]]:
    snapshot = data.copy()
    try:
        yield data
    except Exception:
        data.clear()
        data.update(snapshot)
        raise

inventory = {'A': 8, 'B': 2}
with dictionary_transaction(inventory) as working:
    working['A'] -= 3
    working['B'] += 3


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [15]:
assert inventory == {'A': 5, 'B': 5}
original = inventory.copy()
with CHECK.assertRaisesRegex(RuntimeError, 'payment failed'):
    with dictionary_transaction(inventory) as working:
        working['A'] = 0
        working['NEW'] = 100
        raise RuntimeError('payment failed')
assert inventory == original
with CHECK.assertRaisesRegex(ZeroDivisionError, 'division by zero'):
    with dictionary_transaction(inventory) as working:
        working['B'] = -1
        1 / 0
assert inventory == original
passed(7)


PASS — problem 07


**Further challenge.** Make a nested transaction: reason about when a shallow snapshot cannot undo in-place mutations of nested lists.

## Problem 08 — Raise a read-only AttributeError correctly

**Challenge.** Model a bank account whose account ID is immutable but whose label can be changed. Reassignment of `account_id` must raise a custom exception that is also catchable by an ordinary `except AttributeError`. Do not allow the setter to mutate private state first.

**Solution approach and design choices.** A read-only attribute is naturally an `AttributeError` subtype. This is a legitimate case for subclassing one relevant built-in exception directly. The underscore field is a convention, not a security barrier; for strong immutability, use a different object design.

In [16]:
class ReadOnlyAccountError(AttributeError):
    """An account identifier cannot be changed after construction."""

class BankAccount:
    def __init__(self, account_id: str, label: str) -> None:
        if not account_id:
            raise ValueError('account ID must not be blank')
        self._account_id = account_id
        self.label = label

    @property
    def account_id(self) -> str:
        return self._account_id

    @account_id.setter
    def account_id(self, attempted: str) -> None:
        raise ReadOnlyAccountError('account_id is read-only')

account = BankAccount('AC-42', 'Savings')


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [17]:
assert account.account_id == 'AC-42'
account.label = 'Emergency savings'
assert account.label == 'Emergency savings'
with CHECK.assertRaises(ReadOnlyAccountError):
    account.account_id = 'AC-99'
assert account.account_id == 'AC-42'
assert issubclass(ReadOnlyAccountError, AttributeError)
with CHECK.assertRaises(AttributeError):
    account.account_id = 'ANOTHER'
passed(8)


PASS — problem 08


## Problem 09 — Map an API exception hierarchy to consistent HTTP responses

**Challenge.** Design API exceptions for invalid input (400), missing authentication (401), insufficient authorization (403), missing resources (404), duplicate resources (409), and unexpected server failure (500). Every simulated endpoint call must return the SAME outer shape: `(HTTP status integer, JSON-compatible dictionary)`. Server-internal details must not appear in the response.

**Solution approach and design choices.** Keep the domain failure distinct from HTTP presentation, but give each typed error a stable public code and status. The boundary performs mapping and produces a consistent response. HTTP 401 refers to authentication; a known authenticated identity with inadequate access maps to HTTP 403. Handle a server failure with an intentionally generic public message.

In [18]:
from http import HTTPStatus

class ServiceError(Exception):
    """Base for predictable errors in this mock service."""
    status = HTTPStatus.INTERNAL_SERVER_ERROR
    code = 'internal_error'
    public_message = 'An unexpected server error occurred.'

class BadInput(ServiceError):
    status, code, public_message = HTTPStatus.BAD_REQUEST, 'bad_input', 'Invalid input.'

class MissingAuthentication(ServiceError):
    status, code, public_message = HTTPStatus.UNAUTHORIZED, 'unauthenticated', 'Sign in first.'

class PermissionDenied(ServiceError):
    status, code, public_message = HTTPStatus.FORBIDDEN, 'forbidden', 'Access denied.'

class ResourceMissing(ServiceError):
    status, code, public_message = HTTPStatus.NOT_FOUND, 'not_found', 'Resource not found.'

class DuplicateResource(ServiceError):
    status, code, public_message = HTTPStatus.CONFLICT, 'conflict', 'Resource already exists.'

class StorageFailure(ServiceError):
    """The message passed to Exception is for internal use only."""

def get_mock_record(scenario: str) -> dict[str, str]:
    failures = {
        'bad': BadInput, 'anonymous': MissingAuthentication,
        'denied': PermissionDenied, 'missing': ResourceMissing,
        'duplicate': DuplicateResource, 'storage': StorageFailure,
    }
    if scenario in failures:
        raise failures[scenario]('private trace: db-node-7')
    return {'id': 'R-1', 'value': 'ok'}

def endpoint(scenario: str) -> tuple[int, dict[str, object]]:
    try:
        result = get_mock_record(scenario)
    except ServiceError as exc:
        return int(exc.status), {
            'error': {'code': exc.code, 'message': exc.public_message}
        }
    else:
        return int(HTTPStatus.OK), {'data': result}


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [19]:
expected = {
    'bad': 400, 'anonymous': 401, 'denied': 403,
    'missing': 404, 'duplicate': 409, 'storage': 500,
}
for scenario, status in expected.items():
    actual_status, payload = endpoint(scenario)
    assert actual_status == status
    assert set(payload) == {'error'}
    assert set(payload['error']) == {'code', 'message'}
    assert 'private trace' not in str(payload)
status, payload = endpoint('ok')
assert status == 200 and payload == {'data': {'id': 'R-1', 'value': 'ok'}}
passed(9)


PASS — problem 09


**Further challenge.** For a real web framework, return a response object with the same numeric HTTP status and a JSON body; do not return a JSON-encoded string *inside* an additional JSON response.

## Problem 10 — Log the traceback centrally and redact secrets

**Challenge.** Create a named application logger and an in-memory log handler. Catch a predictable payment failure once, log a safe message with the traceback, and return a stable error code. Prove that the log contains exception information but does not contain a private card token.

**Solution approach and design choices.** Exceptions should carry error facts; a central boundary owns logging. `logger.exception()` attaches traceback and must be called within an active `except` block. Do not include secrets in either the exception's string representation or log-format arguments. Production logs may require additional redaction controls.

In [20]:
import logging

class MemoryLogHandler(logging.Handler):
    """Capture log records for deterministic, I/O-free tests."""
    def __init__(self) -> None:
        super().__init__()
        self.records: list[logging.LogRecord] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.records.append(record)

payment_logger = logging.getLogger('custom_exercises.payment')
payment_logger.setLevel(logging.ERROR)
payment_logger.propagate = False
payment_handler = MemoryLogHandler()
payment_logger.handlers.clear()  # Notebook reruns should not duplicate handlers.
payment_logger.addHandler(payment_handler)

class PaymentGatewayError(Exception):
    """Safe, non-sensitive payment gateway failure."""

def gateway_charge(secret_token: str) -> None:
    if secret_token != 'accepted':
        raise PaymentGatewayError('gateway operation rejected')

def payment_boundary(secret_token: str) -> str:
    try:
        gateway_charge(secret_token)
    except PaymentGatewayError:
        payment_logger.exception('Payment operation failed; code=PAYMENT_GATEWAY')
        return 'payment_unavailable'
    return 'paid'


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [21]:
private_token = 'card-token-DO-NOT-LOG'
assert payment_boundary(private_token) == 'payment_unavailable'
assert len(payment_handler.records) == 1
record = payment_handler.records[0]
assert record.levelno == logging.ERROR
assert record.exc_info is not None
assert 'PAYMENT_GATEWAY' in record.getMessage()
assert private_token not in record.getMessage()
assert private_token not in str(record.exc_info[1])
assert payment_boundary('accepted') == 'paid'
assert len(payment_handler.records) == 1
passed(10)


PASS — problem 10


## Problem 11 — Design a stable JSON error contract

**Challenge.** Build an exception with a machine-readable code, validated HTTP status, safe public text, and a request ID. Provide `to_dict()` and `to_json()`; verify correct round-tripping and that changing the returned dict does not mutate the exception. Reject illegal statuses and non-string request IDs.

**Solution approach and design choices.** Pass plain JSON primitives to `json.dumps`; status is explicitly an integer. Expose only known public fields. Returning a new nested dictionary each time prevents accidental mutation of exception state. Real request IDs should be generated at a trusted boundary rather than accepted blindly from untrusted headers.

In [22]:
import json

class PublicAPIError(Exception):
    """An error with a safe, transport-ready public representation."""

    def __init__(self, code: str, message: str, status: int, request_id: str) -> None:
        if not code or not isinstance(code, str):
            raise ValueError('code must be a nonempty string')
        if type(status) is not int or not 400 <= status <= 599:
            raise ValueError('status must be an integer HTTP error status')
        if not isinstance(message, str) or not message:
            raise ValueError('message must be a nonempty string')
        if not isinstance(request_id, str):
            raise TypeError('request_id must be a string')
        self.code = code
        self.status = status
        self.public_message = message
        self.request_id = request_id
        super().__init__(f'{code} ({status})')

    def to_dict(self) -> dict[str, object]:
        return {
            'error': {
                'code': self.code,
                'message': self.public_message,
                'request_id': self.request_id,
            }
        }

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), ensure_ascii=False, sort_keys=True)

public_error = PublicAPIError('not_found', 'Resource not found.', 404, 'req-001')


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [23]:
parsed = json.loads(public_error.to_json())
assert parsed == public_error.to_dict()
assert parsed['error']['code'] == 'not_found'
assert public_error.status == 404
parsed['error']['message'] = 'MUTATED'
assert public_error.public_message == 'Resource not found.'
for bad_status in [200, 600, True, 404.0]:
    with CHECK.assertRaises(ValueError):
        PublicAPIError('bad', 'Bad input', bad_status, 'req')
with CHECK.assertRaises(TypeError):
    PublicAPIError('bad', 'Bad input', 400, None)
passed(11)


PASS — problem 11


**Further challenge.** Consider whether a correlation/request ID is safe to reveal in your particular threat model; its mere presence does not replace secure server-side logging.

## Problem 12 — Implement bounded retries on transient exceptions only

**Challenge.** Implement `retry_transient(operation, attempts, sleep)` with injected sleep for deterministic testing. Retry only `TemporaryNetworkFailure`, not `PermanentNetworkFailure`; reject attempts below one and booleans; use exponential delays capped at 8 seconds; preserve the final exception type and identity when retries are exhausted.

**Solution approach and design choices.** Exception types make retryability explicit. Catch only a transient leaf, never arbitrary `Exception`, and use a bare `raise` after the final failure to preserve its traceback. Backoff is deterministic here; a production distributed client usually adds jitter and respects timeouts, server retry hints, and idempotency.

In [24]:
from collections.abc import Callable
from typing import TypeVar

T = TypeVar('T')

class NetworkOperationError(Exception):
    """Root of predictable operation failures."""

class TemporaryNetworkFailure(NetworkOperationError):
    """Retryable network failure."""

class PermanentNetworkFailure(NetworkOperationError):
    """Do not retry this failure."""

def retry_transient(operation: Callable[[], T], attempts: int,
                    sleep: Callable[[float], None]) -> T:
    if type(attempts) is not int or attempts < 1:
        raise ValueError('attempts must be a positive integer')
    for index in range(attempts):
        try:
            return operation()
        except TemporaryNetworkFailure:
            if index + 1 == attempts:
                raise
            sleep(min(2 ** index, 8))
    raise AssertionError('unreachable')


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [25]:
calls, delays = [], []
def flaky() -> str:
    calls.append('attempt')
    if len(calls) < 4:
        raise TemporaryNetworkFailure('short outage')
    return 'ok'
assert retry_transient(flaky, 5, delays.append) == 'ok'
assert len(calls) == 4 and delays == [1, 2, 4]
original_error = TemporaryNetworkFailure('still down')
attempts_made = []
def always_down() -> None:
    attempts_made.append(1)
    raise original_error
with CHECK.assertRaises(TemporaryNetworkFailure) as captured:
    retry_transient(always_down, 3, lambda _: None)
assert captured.exception is original_error and len(attempts_made) == 3
permanent_calls = []
def permanent() -> None:
    permanent_calls.append(1)
    raise PermanentNetworkFailure('invalid host')
with CHECK.assertRaises(PermanentNetworkFailure):
    retry_transient(permanent, 5, delays.append)
assert len(permanent_calls) == 1
for invalid in [0, -1, True]:
    with CHECK.assertRaises(ValueError):
        retry_transient(lambda: None, invalid, delays.append)
passed(12)


PASS — problem 12


## Problem 13 — Partition batched failures with ExceptionGroup

**Challenge.** Validate several independent records, collect domain exceptions instead of stopping at the first one, and raise `ExceptionGroup`. Handle `ValueError`-based and `LookupError`-based failures with separate `except*` clauses. Verify successful rows are retained and all errors are counted.

**Solution approach and design choices.** `ExceptionGroup` (Python 3.11+) represents multiple concurrent or batched failures. An `except*` clause handles its matching *subgroup*; it is not a normal `except`. Avoid flattening to strings until you have handled typed failures. Never silently discard unhandled error types.

In [26]:
class BatchValueError(ValueError):
    """A record contains an invalid value."""

class BatchLookupError(LookupError):
    """A record refers to an unknown key."""

def validate_batch(rows: list[dict[str, object]]) -> list[int]:
    valid: list[int] = []
    failures: list[Exception] = []
    for position, row in enumerate(rows):
        if 'quantity' not in row:
            failures.append(BatchLookupError(f'row {position}: quantity missing'))
        elif type(row['quantity']) is not int or row['quantity'] <= 0:
            failures.append(BatchValueError(f'row {position}: quantity invalid'))
        else:
            valid.append(row['quantity'])
    if failures:
        raise ExceptionGroup('batch validation failed', failures)
    return valid


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [27]:
assert validate_batch([{'quantity': 1}, {'quantity': 3}]) == [1, 3]
matched_types: list[str] = []
try:
    validate_batch([{'quantity': 1}, {}, {'quantity': -1}, {'quantity': True}])
except* BatchLookupError as grouped:
    matched_types.extend(type(error).__name__ for error in grouped.exceptions)
except* BatchValueError as grouped:
    matched_types.extend(type(error).__name__ for error in grouped.exceptions)
assert matched_types.count('BatchLookupError') == 1
assert matched_types.count('BatchValueError') == 2
with CHECK.assertRaises(ExceptionGroup) as captured:
    validate_batch([{}, {'quantity': 0}])
assert len(captured.exception.exceptions) == 2
passed(13)


PASS — problem 13


**Further challenge.** Try nested `ExceptionGroup` objects and investigate `.subgroup()` / `.split()` to preserve nested structure.

## Problem 14 — Keep successful concurrent results while collecting typed failures

**Challenge.** Run independent simulated jobs through `ThreadPoolExecutor`. Return successes and an `ExceptionGroup` of known job errors without losing successful results. Preserve job IDs inside custom exceptions, and ensure output ordering is deterministic despite asynchronous completion.

**Solution approach and design choices.** A future stores its failure and raises it when `future.result()` is retrieved. Catch the narrow known job exception, collect it, and use a stable sort on results for reproducible tests. In production, distinguish partial-success batch APIs from all-or-nothing transaction APIs.

In [28]:
from concurrent.futures import ThreadPoolExecutor, as_completed

class WorkerJobError(Exception):
    """An expected job-specific failure."""
    def __init__(self, job_id: int, message: str) -> None:
        self.job_id = job_id
        super().__init__(f'job {job_id}: {message}')

def worker(job_id: int) -> tuple[int, int]:
    if job_id % 3 == 0:
        raise WorkerJobError(job_id, 'simulated worker failure')
    return job_id, job_id * job_id

def execute_jobs(ids: list[int]) -> tuple[list[tuple[int, int]], ExceptionGroup | None]:
    success: list[tuple[int, int]] = []
    failures: list[Exception] = []
    with ThreadPoolExecutor(max_workers=3) as pool:
        future_to_id = {pool.submit(worker, job_id): job_id for job_id in ids}
        for future in as_completed(future_to_id):
            try:
                success.append(future.result())
            except WorkerJobError as error:
                failures.append(error)
    success.sort(key=lambda pair: pair[0])
    failures.sort(key=lambda error: error.job_id)
    grouped = ExceptionGroup('worker failures', failures) if failures else None
    return success, grouped


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [29]:
successes, failures = execute_jobs([1, 2, 3, 4, 5, 6])
assert successes == [(1, 1), (2, 4), (4, 16), (5, 25)]
assert isinstance(failures, ExceptionGroup)
assert [err.job_id for err in failures.exceptions] == [3, 6]
all_success, no_errors = execute_jobs([1, 2, 4])
assert len(all_success) == 3 and no_errors is None
assert execute_jobs([]) == ([], None)
passed(14)


PASS — problem 14


## Problem 15 — Preserve asyncio task cancellation

**Challenge.** Write an asynchronous worker that records cleanup but never converts `asyncio.CancelledError` into a normal success. Cancel an in-progress task and prove the cancellation reaches its caller. Show how a separate, expected async operation failure can be translated without trapping cancellation.

**Solution approach and design choices.** Cancellation is a control-flow signal; propagate it after cleanup. Modern Python's `asyncio.CancelledError` inherits directly from `BaseException`, not `Exception`, so even a broad `except Exception` would not ordinarily trap it, but explicitly handling cancellation is useful when cleanup is needed. Jupyter supports top-level `await`; do not nest `asyncio.run()` inside an active notebook event loop.

In [30]:
import asyncio

class AsyncServiceFailure(Exception):
    """Predictable application-level failure from an async service."""

async def cancellable_worker(events: list[str]) -> str:
    try:
        events.append('started')
        await asyncio.sleep(3600)
        return 'completed'
    except asyncio.CancelledError:
        events.append('cancelled')
        raise
    finally:
        events.append('cleanup')

async def translated_async_call(should_fail: bool) -> str:
    try:
        await asyncio.sleep(0)
        if should_fail:
            raise OSError('private upstream unavailable')
        return 'ready'
    except OSError as cause:
        raise AsyncServiceFailure('Async service unavailable') from cause

async def check_async_problem() -> None:
    events: list[str] = []
    task = asyncio.create_task(cancellable_worker(events))
    await asyncio.sleep(0)  # Let the worker start before requesting cancellation.
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        events.append('caller-observed-cancellation')
    else:
        raise AssertionError('The cancelled task must not report success')
    assert events == ['started', 'cancelled', 'cleanup', 'caller-observed-cancellation']
    assert await translated_async_call(False) == 'ready'
    with CHECK.assertRaises(AsyncServiceFailure) as captured:
        await translated_async_call(True)
    assert isinstance(captured.exception.__cause__, OSError)


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [31]:
await check_async_problem()
assert issubclass(asyncio.CancelledError, BaseException)
assert not issubclass(asyncio.CancelledError, Exception)
passed(15)


PASS — problem 15


**Further challenge.** Add a timeout using `asyncio.timeout()` and explain why a timeout and cancellation have different recovery semantics.

## Problem 16 — Make a custom exception safely pickleable

**Challenge.** A task queue may serialize exceptions for local diagnostic storage. Create a custom exception with a required `task_id` and `reason`, implement correct pickling and unpacking, and verify the round trip preserves fields, type, args, and message. Use only trusted serialized data in this exercise.

**Solution approach and design choices.** Exceptions with nontrivial constructor signatures may need an explicit reconstruction contract. `__reduce__` returns the callable and its required construction arguments. **Never unpickle untrusted input:** pickle can execute arbitrary code. Prefer JSON with an allowlisted schema for network or user-controlled data.

In [32]:
import pickle

class TaskProcessingError(Exception):
    """A task failed with structured diagnostic fields."""

    def __init__(self, task_id: int, reason: str) -> None:
        if type(task_id) is not int:
            raise TypeError('task_id must be an integer')
        self.task_id = task_id
        self.reason = reason
        super().__init__(f'Task {task_id} failed: {reason}')

    def __reduce__(self):
        return (type(self), (self.task_id, self.reason))

original_task_error = TaskProcessingError(17, 'disk unavailable')


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [33]:
restored = pickle.loads(pickle.dumps(original_task_error))
assert type(restored) is TaskProcessingError
assert restored is not original_task_error
assert restored.task_id == 17 and restored.reason == 'disk unavailable'
assert restored.args == original_task_error.args
assert str(restored) == str(original_task_error)
with CHECK.assertRaises(TypeError):
    TaskProcessingError(True, 'not an integer ID')
passed(16)


PASS — problem 16


## Problem 17 — Provide compatibility without fragile exception multiple inheritance

**Challenge.** You are replacing a function that historically raised `ValueError` with a library whose public root is `CalculationError`. Show a domain-specific error under that root and an optional compatibility wrapper that translates it back to `ValueError` with a preserved cause. Explain why automatic inheritance from both roots is not generally the safest default.

**Solution approach and design choices.** Many built-in exception classes use custom C layouts; inheriting from multiple exception types can cause conflicts or subtle version-dependent behavior. A single application hierarchy is simpler. At a specifically documented compatibility boundary, translate a domain error to a built-in error using `from`, rather than trying to make one class occupy unrelated hierarchies simultaneously.

In [34]:
class CalculationError(Exception):
    """Base for the new calculations library."""

class NegativeOperandError(CalculationError):
    """A nonnegative operand was required."""
    def __init__(self, value: int) -> None:
        self.value = value
        super().__init__(f'operand cannot be negative: {value}')

def new_square_root_input(value: int) -> int:
    if type(value) is not int:
        raise TypeError('value must be an integer')
    if value < 0:
        raise NegativeOperandError(value)
    return value

def legacy_square_root_input(value: int) -> int:
    try:
        return new_square_root_input(value)
    except NegativeOperandError as original:
        raise ValueError(str(original)) from original


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [35]:
assert new_square_root_input(0) == 0
with CHECK.assertRaises(CalculationError) as captured:
    new_square_root_input(-4)
assert type(captured.exception) is NegativeOperandError
assert not isinstance(captured.exception, ValueError)
with CHECK.assertRaises(ValueError) as captured:
    legacy_square_root_input(-4)
assert isinstance(captured.exception.__cause__, NegativeOperandError)
with CHECK.assertRaises(TypeError):
    legacy_square_root_input(True)
passed(17)


PASS — problem 17


**Further challenge.** For a breaking public library release, write migration documentation describing the precise exception contract rather than silently translating everything.

## Problem 18 — Implement different recovery policies for optional and critical plugins

**Challenge.** A plugin may be temporarily unavailable or may return malformed data. An optional enrichment feature may recover from *unavailability* by using a fallback; a security-critical permission check must fail closed. Unexpected protocol failures should propagate instead of being disguised as unavailability.

**Solution approach and design choices.** Recovery depends on what the operation means, not just on whether an exception is catchable. Catch a narrow `PluginUnavailable` for optional functionality; do not catch `Exception` and return success. Security checks must never treat plugin failure as permission granted. Malformed data is a separate error type.

In [36]:
class PluginFailure(Exception):
    """Root for anticipated plugin failures."""

class PluginUnavailable(PluginFailure):
    """A plugin cannot currently be reached."""

class PluginProtocolError(PluginFailure):
    """A plugin responded in an invalid format."""

def plugin_call(mode: str) -> str:
    if mode == 'offline':
        raise PluginUnavailable('plugin is offline')
    if mode == 'bad-format':
        raise PluginProtocolError('invalid plugin response')
    return 'plugin-result'

def optional_enrichment(mode: str) -> str:
    try:
        return plugin_call(mode)
    except PluginUnavailable:
        return 'default-enrichment'

def may_access_resource(mode: str) -> bool:
    try:
        return plugin_call(mode) == 'plugin-result'
    except PluginUnavailable:
        return False  # Fail closed, never grant access on plugin failure.


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [37]:
assert optional_enrichment('ok') == 'plugin-result'
assert optional_enrichment('offline') == 'default-enrichment'
with CHECK.assertRaises(PluginProtocolError):
    optional_enrichment('bad-format')
assert may_access_resource('ok') is True
assert may_access_resource('offline') is False
with CHECK.assertRaises(PluginProtocolError):
    may_access_resource('bad-format')
passed(18)


PASS — problem 18


## Problem 19 — Create deterministic boundary and fuzz-style tests

**Challenge.** Implement strict division of a numerator by a denominator with domain errors for zero denominators and values outside ±100. Test known boundaries, invalid types (including bool), and 250 pseudo-random deterministic integer pairs. For every valid pair, check the mathematical result; for invalid pairs, check exact expected error families.

**Solution approach and design choices.** Exception correctness benefits from invariant and boundary tests, not just one happy path. Use a local seeded PRNG for reproducibility without altering global randomness. Catch narrowly in tests and fail immediately on any disagreement between the oracle and implementation.

In [38]:
import random

class DivisionDomainError(ValueError):
    """The input is numeric but not permitted by this API."""

class ZeroDenominatorError(DivisionDomainError):
    """Division by zero is not allowed."""

def strict_divide(numerator: object, denominator: object) -> float:
    if type(numerator) is not int or type(denominator) is not int:
        raise TypeError('both operands must be integers, not bool')
    if not -100 <= numerator <= 100 or not -100 <= denominator <= 100:
        raise DivisionDomainError('operands must lie between -100 and 100')
    if denominator == 0:
        raise ZeroDenominatorError('denominator must not be zero')
    return numerator / denominator


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [39]:
assert strict_divide(100, -100) == -1.0
assert strict_divide(0, 7) == 0.0
for bad in [(True, 1), (2, False), (1.2, 3), (1, None)]:
    with CHECK.assertRaises(TypeError):
        strict_divide(*bad)
for bad in [(101, 2), (-101, 2), (2, -101)]:
    with CHECK.assertRaises(DivisionDomainError):
        strict_divide(*bad)
with CHECK.assertRaises(ZeroDenominatorError):
    strict_divide(12, 0)
rng = random.Random(20260922)
case_count = 0
for _ in range(250):
    numerator = rng.randint(-110, 110)
    denominator = rng.randint(-110, 110)
    if not (-100 <= numerator <= 100 and -100 <= denominator <= 100):
        with CHECK.assertRaises(DivisionDomainError):
            strict_divide(numerator, denominator)
    elif denominator == 0:
        with CHECK.assertRaises(ZeroDenominatorError):
            strict_divide(numerator, denominator)
    else:
        assert strict_divide(numerator, denominator) == numerator / denominator
    case_count += 1
assert case_count == 250
passed(19)


PASS — problem 19


## Problem 20 — Capstone: atomic order service, typed errors, safe API mapping, bounded retry

**Challenge.** Build a complete in-memory order system with stock, order IDs, quantity validation, authorization, and a simulated transient database write failure. Requirements: (1) reject invalid quantities including bool; (2) deny unauthenticated/unauthorized access without leaking details; (3) distinguish duplicates and stock conflicts; (4) roll back BOTH inventory and orders for a failed write; (5) retry only transient writes with a bounded count; (6) return `(status, JSON-compatible dict)` on every API path; and (7) test success, rollback, exhaustion, conflicts, error privacy, and state invariants.

**Solution approach and design choices.** Use one clear domain exception hierarchy. Validate before mutating; snapshot both stores to simulate an atomic transaction; restore them and re-raise on known failures; retry only a transient subclass. The simulated transient fault occurs *after decrement* to prove rollback. The retry is safe here only because the failed operation is fully rolled back; production systems require database transactions and robust idempotency keys. Keep public messages on the presentation boundary. `401` vs `403` follow their distinct HTTP meanings.

In [40]:
from dataclasses import dataclass
from http import HTTPStatus

class OrderError(Exception):
    """Base class for predictable order-service errors."""

class OrderInputError(OrderError):
    """Quantity or identifier is invalid."""

class OrderUnauthenticated(OrderError):
    """No authenticated account is available."""

class OrderForbidden(OrderError):
    """Account is known but cannot place orders."""

class OrderConflict(OrderError):
    """An order ID or stock constraint conflicts with state."""

class OrderTransientWriteError(OrderError):
    """An expected, temporary storage write failure."""

@dataclass(frozen=True)
class OrderRecord:
    order_id: str
    sku: str
    quantity: int

class OrderService:
    """Small, single-process teaching model; NOT a concurrent real database."""
    def __init__(self, stock: dict[str, int], write_failures: int = 0) -> None:
        self.stock = stock.copy()
        self.orders: dict[str, OrderRecord] = {}
        self.write_failures = write_failures

    def place(self, order_id: str, sku: str, quantity: int,
              authenticated: bool = True, authorized: bool = True) -> OrderRecord:
        if not authenticated:
            raise OrderUnauthenticated('credentials required')
        if not authorized:
            raise OrderForbidden('account lacks purchase permission')
        if not isinstance(order_id, str) or not order_id:
            raise OrderInputError('order_id must be a nonempty string')
        if not isinstance(sku, str) or not sku:
            raise OrderInputError('sku must be a nonempty string')
        if type(quantity) is not int or quantity <= 0:
            raise OrderInputError('quantity must be a positive integer')
        if order_id in self.orders:
            raise OrderConflict('duplicate order ID')
        if quantity > self.stock.get(sku, 0):
            raise OrderConflict('insufficient inventory')

        stock_before = self.stock.copy()
        orders_before = self.orders.copy()
        try:
            self.stock[sku] -= quantity  # Mutation happens BEFORE the simulated fault.
            if self.write_failures:
                self.write_failures -= 1
                raise OrderTransientWriteError('private storage node db-07 offline')
            record = OrderRecord(order_id, sku, quantity)
            self.orders[order_id] = record
            return record
        except OrderError:
            self.stock.clear()
            self.stock.update(stock_before)
            self.orders.clear()
            self.orders.update(orders_before)
            raise

def place_order_with_retry(service: OrderService, *, order_id: str, sku: str,
                           quantity: int, authenticated: bool = True,
                           authorized: bool = True, attempts: int = 3) -> OrderRecord:
    if type(attempts) is not int or attempts < 1:
        raise ValueError('attempts must be a positive integer')
    for index in range(attempts):
        try:
            return service.place(order_id, sku, quantity, authenticated, authorized)
        except OrderTransientWriteError:
            if index + 1 == attempts:
                raise
    raise AssertionError('unreachable')

ORDER_PUBLIC = {
    OrderInputError: (HTTPStatus.BAD_REQUEST, 'bad_order', 'Invalid order input.'),
    OrderUnauthenticated: (HTTPStatus.UNAUTHORIZED, 'unauthenticated', 'Sign in first.'),
    OrderForbidden: (HTTPStatus.FORBIDDEN, 'forbidden', 'Access denied.'),
    OrderConflict: (HTTPStatus.CONFLICT, 'conflict', 'Order cannot be completed.'),
    OrderTransientWriteError: (
        HTTPStatus.SERVICE_UNAVAILABLE, 'temporarily_unavailable',
        'The service is temporarily unavailable.'
    ),
}

def order_endpoint(service: OrderService, *, order_id: str, sku: str, quantity: int,
                   authenticated: bool = True, authorized: bool = True,
                   attempts: int = 3) -> tuple[int, dict[str, object]]:
    try:
        record = place_order_with_retry(
            service, order_id=order_id, sku=sku, quantity=quantity,
            authenticated=authenticated, authorized=authorized, attempts=attempts,
        )
    except OrderError as error:
        status, code, public_message = ORDER_PUBLIC[type(error)]
        return int(status), {'error': {'code': code, 'message': public_message}}
    else:
        return int(HTTPStatus.CREATED), {
            'data': {'order_id': record.order_id, 'sku': record.sku,
                     'quantity': record.quantity}
        }


**Executable checks.** Run this cell after the solution. Assertions deliberately include positive, negative, and edge cases where relevant.

In [41]:
service = OrderService({'BOOK': 5}, write_failures=1)
status, body = order_endpoint(service, order_id='O-1', sku='BOOK', quantity=2)
assert (status, body) == (201, {'data': {'order_id': 'O-1', 'sku': 'BOOK', 'quantity': 2}})
assert service.stock == {'BOOK': 3}
assert service.orders == {'O-1': OrderRecord('O-1', 'BOOK', 2)}
assert service.write_failures == 0  # One rolled-back attempt, then successful retry.

before_stock, before_orders = service.stock.copy(), service.orders.copy()
scenarios = [
    ({'order_id': 'O-1', 'sku': 'BOOK', 'quantity': 1}, 409),
    ({'order_id': 'O-2', 'sku': 'BOOK', 'quantity': 99}, 409),
    ({'order_id': 'O-2', 'sku': 'BOOK', 'quantity': True}, 400),
    ({'order_id': 'O-2', 'sku': 'BOOK', 'quantity': 0}, 400),
    ({'order_id': 'O-2', 'sku': 'BOOK', 'quantity': 1,
      'authenticated': False}, 401),
    ({'order_id': 'O-2', 'sku': 'BOOK', 'quantity': 1,
      'authorized': False}, 403),
]
for kwargs, expected_status in scenarios:
    status, body = order_endpoint(service, **kwargs)
    assert status == expected_status
    assert set(body) == {'error'}
    assert service.stock == before_stock and service.orders == before_orders

exhausted = OrderService({'BOOK': 4}, write_failures=5)
status, body = order_endpoint(
    exhausted, order_id='O-X', sku='BOOK', quantity=3, attempts=2
)
assert status == 503
assert body['error']['code'] == 'temporarily_unavailable'
assert 'private storage node' not in str(body)
assert exhausted.stock == {'BOOK': 4} and exhausted.orders == {}
assert exhausted.write_failures == 3
with CHECK.assertRaises(ValueError):
    place_order_with_retry(exhausted, order_id='X', sku='BOOK', quantity=1, attempts=True)
passed(20)


PASS — problem 20


**Further challenge.** Extend this design with an idempotency key and a real database transaction. Consider concurrency: the in-memory snapshots here are not safe against multiple simultaneous writers.

---
## Consolidated self-check and study notes

The notebook's individual cells exercise expected success, exception identity, type hierarchies, public-message safety, failure recovery, deterministic concurrency, and asynchronous cancellation. The final assertion below confirms that all **20** dedicated problem test cells ran in order. It is a smoke-check for the notebook execution sequence, not a substitute for real deployment, framework integration tests, or security review.

**Review questions:** What should a caller catch: the leaf, branch, or library root? When is exception translation valuable? What does `raise ... from None` hide, and what does it *not* erase? Why is a 403 different from a 401? When is retry unsafe without idempotency? How should cancellation and grouped failures differ from ordinary domain errors?

In [42]:
assert PASS_COUNT == 20, f'Expected 20 passing problem checks; observed {PASS_COUNT}'
print(f'ALL CHECKS PASSED: {PASS_COUNT}/20 advanced problems')


ALL CHECKS PASSED: 20/20 advanced problems
